# ShopDesk, Module 2 Section 1 Lab 1: Refining Tool Descriptions

A beginner-friendly notebook on **tool selection**. We give ShopDesk four tools with
**overlapping, vague** descriptions, watch the model mis-route ambiguous queries, then
**refine** the descriptions until each tool owns a clear slice, and **measure** the
selection accuracy before and after. Built on the **base Anthropic SDK**, running **Sonnet**
(`claude-sonnet-4-6`) through your **Anthropic API key**.

## The real-world scenario

When ShopDesk has several tools, the model does not read your code to choose one: it reads
the **descriptions**. If two tools both say "get order information," the model has no honest
way to pick, so it routes at random and sometimes runs the wrong action. The fix is not more
code; it is sharper descriptions with clear boundaries.

The question this lab answers: **how much does the wording of a tool description change which
tool the model picks, and how do you write descriptions that route cleanly?**

## Objectives

- See that the **description** is the primary signal the model uses to select a tool.
- Rewrite overlapping descriptions to include **boundaries**: input format, example queries,
  edge cases, and when NOT to use the tool.
- **Measure** selection accuracy on ambiguous queries before and after refinement.

## What you'll observe

- A pure-Python overlap score is high for the vague tools and low for the refined ones.
- Live, the vague tool set mis-routes several ambiguous queries; the refined set routes them
  correctly.
- The only thing that changed between the two runs is the description text.

## How to run

Run top to bottom. The tool definitions and the overlap score are pure Python and run
anywhere. The selection-accuracy cell calls Claude, so paste a real key into **Setup 2/3**
and re-run from the top; otherwise it skips. This lab uses the base Messages API so you can
read exactly which tool the model chose.

## 0. Setup

**This cell:** installs the packages. This lab uses only the **base Anthropic SDK**,
because tool selection is a property of one Messages API call: you pass `tools` and read back
which one the model picked.

In [ ]:
# ===== SETUP 1/3 - install the base SDK =====
%pip install -q anthropic python-dotenv

**This cell:** imports, the model, and the `RUN_LIVE` switch so the selection call fires
only with a real key.

In [ ]:
# ===== SETUP 2/3 - imports, the model, and a live/offline switch =====
import os                                       # read the API key from the environment
import re                                       # tokenise descriptions for the overlap score
import itertools                                # pair tools up for the overlap score
import anthropic                                # the base Anthropic SDK (synchronous)

try:                                            # load a .env file if present
    from dotenv import load_dotenv              #   import the loader
    load_dotenv()                               #   read .env into environment variables
except Exception:                               # not installed? that is fine
    pass                                        #   set the key another way

MODEL = "claude-sonnet-4-6"                      # the Sonnet model every call will use

os.environ.setdefault("ANTHROPIC_API_KEY", "sk-ant-...")     # placeholder unless you set a real key
_key = os.environ["ANTHROPIC_API_KEY"]           # read whatever key is set
RUN_LIVE = _key.startswith("sk-ant-") and _key != "sk-ant-..."   # True only for a real key
print("live model calls:", "ON" if RUN_LIVE else "OFF (using a placeholder key)")

**This cell:** a small shared input schema. All four tools take one `order_id`, so the
*only* thing that distinguishes them is the description. Isolating that is the whole point of
the lab.

In [ ]:
# ===== SETUP 3/3 - the shared input schema =====
ARG = {"type": "object",                          # every tool takes one string order_id
       "properties": {"order_id": {"type": "string"}},
       "required": ["order_id"]}
print("shared input schema ready")

### The description is the decision signal

The model chooses a tool by matching the user's request against each tool's **description**,
not its name or code. A good description draws a clear boundary: what the tool is for, the
input it expects, example queries that should trigger it, and, just as important, when NOT to
use it. Overlapping descriptions blur those boundaries and cause mis-routing.

---

### 🎯 Lab objective - sharpen descriptions, measure the gain

**What you build:** two versions of the same four tools (vague and refined), an overlap score,
and an accuracy test over ambiguous queries.

**Why it helps you build real solutions:** description quality is the cheapest, highest-leverage
lever on tool routing. A ten-minute rewrite often fixes what looks like a model problem.

**How you'll see it:** the overlap score drops and the selection accuracy rises, with only the
descriptions changed.

**This cell:** version 1, the **vague** tools. Every description is a generic "get order
information," so they overlap heavily. For a query like "where is A1," three of them look
equally valid, which is exactly the routing problem.

In [ ]:
# ===== v1: four vague, overlapping tools =====
TOOLS_V1 = [                                       # only the descriptions matter here
    {"name": "check_status",  "description": "Get information about an order.",          "input_schema": ARG},
    {"name": "track_package", "description": "Get information about a package or order.", "input_schema": ARG},
    {"name": "order_details", "description": "Get details about an order.",              "input_schema": ARG},
    {"name": "refund_lookup", "description": "Look up order information related to money.","input_schema": ARG},
]
for t in TOOLS_V1:                                 # show how similar they read
    print(f"  {t['name']:14} {t['description']}")

**This cell:** version 2, the **refined** tools. Each description now states its slice,
the input, example queries, and when NOT to use it. The names and schemas are unchanged; only
the wording is sharper, so each tool owns a distinct job.

In [ ]:
# ===== v2: the same four tools, with clear boundaries =====
TOOLS_V2 = [
    {"name": "check_status",
     "description": ("Return the lifecycle state (processing, shipped, or delivered) of ONE order by id. "
                     "Use for 'where is my order', 'has A1 shipped', 'is it delivered yet'. "
                     "Input: order_id like 'A1'. Do NOT use for tracking links, order contents, or refunds."),
     "input_schema": ARG},
    {"name": "track_package",
     "description": ("Return the carrier tracking URL for a shipped order. Use ONLY when the customer "
                     "explicitly asks for a tracking link or number, e.g. 'send me the tracking link for A1'. "
                     "Do NOT use for general 'where is my order' status; use check_status for that."),
     "input_schema": ARG},
    {"name": "order_details",
     "description": ("Return the contents of an order: items, quantities, and total price. Use for "
                     "'what did I order', 'what is in A1', 'how much was A1'. "
                     "Do NOT use for status, tracking, or refunds."),
     "input_schema": ARG},
    {"name": "refund_lookup",
     "description": ("Return whether an order is refundable under the 30-day rule and any refund already "
                     "issued. Use for 'can I get a refund', 'is A2 refundable', 'money back'. "
                     "Do NOT use for shipping status or order contents."),
     "input_schema": ARG},
]
print("refined descriptions ready for:", [t["name"] for t in TOOLS_V2])

**This cell:** an **overlap score**, pure Python. It measures how many content words the
tool descriptions share, pairwise. High overlap means the model cannot tell them apart. This
runs offline and previews what the accuracy test will confirm.

In [ ]:
# ===== measure description overlap (offline proxy for ambiguity) =====
STOP = {"the", "a", "an", "of", "for", "to", "or", "and", "use", "do", "not", "by",
        "on", "in", "with", "is", "any", "one", "return", "get"}   # drop only true filler words

def words(desc):                                   # description -> its set of content words
    return {w for w in re.findall(r"[a-z']+", desc.lower()) if w not in STOP and len(w) > 2}

def overlap_score(tools):                          # average pairwise word overlap (Jaccard)
    sets = [words(t["description"]) for t in tools]
    pairs = list(itertools.combinations(sets, 2))  #   every pair of tools
    jac = [len(a & b) / len(a | b) for a, b in pairs if (a | b)]   # shared / total words
    return round(sum(jac) / len(jac), 3) if jac else 0.0

print("v1 overlap (vague):  ", overlap_score(TOOLS_V1), "  higher = more ambiguous")
print("v2 overlap (refined):", overlap_score(TOOLS_V2), "  lower  = clearer boundaries")

**This cell:** `pick_tool()`, which asks the model to choose a tool for a query. We set
`tool_choice={"type":"any"}` so the model must pick one of the tools, letting us read the
selection directly. This isolates the description's effect on routing.

In [ ]:
# ===== ask the model which tool it would pick =====
def pick_tool(query, tools):                       # query -> the chosen tool name
    client = anthropic.Anthropic()                 #   the LLM client (reads the key)
    r = client.messages.create(                    #   one Messages API call
        model=MODEL, max_tokens=200, tools=tools,   #   hand over the tool set
        tool_choice={"type": "any"},               #   MUST pick a tool -> we can read the choice
        messages=[{"role": "user", "content": query}])
    for b in r.content:                            #   find the tool_use block
        if b.type == "tool_use":
            return b.name                          #     the tool the model selected
    return "(none)"                                #   should not happen with tool_choice any

**This cell:** the **ambiguous query set**, each labelled with the tool that should
handle it. These are the queries a vague description struggles with. This runs offline; the
scoring against it is live.

In [ ]:
# ===== the labelled ambiguous queries =====
EVAL = [                                            # (query, the tool that SHOULD be picked)
    ("Where is order A1 right now?",                  "check_status"),
    ("Has order A1 shipped yet?",                     "check_status"),
    ("Can you send me the tracking link for A1?",     "track_package"),
    ("What items are in order A1 and how much was it?","order_details"),
    ("Can I get a refund on order A2?",               "refund_lookup"),
]
print("evaluation queries:", len(EVAL))

**This cell:** runs the accuracy test on **both** tool sets and prints, per query, the
tool each version picked versus the expected one. The vague set mis-routes; the refined set
should route cleanly, with only the descriptions different.

In [ ]:
# ===== compare selection accuracy: vague vs refined =====
def accuracy(tools, label):                        # run EVAL through pick_tool, print, return score
    correct = 0                                     #   count right selections
    for query, expected in EVAL:                    #   walk every labelled query
        got = pick_tool(query, tools)               #     what the model picked
        ok = (got == expected)                      #     right?
        correct += ok
        print(f"  {label:8} {query[:38]!r:40} picked={got:14} want={expected:14} {'ok' if ok else 'X'}")
    print(f"  {label} accuracy: {correct}/{len(EVAL)}\n")
    return correct

if RUN_LIVE:                                        # needs a real key
    print("v1 (vague):");   a1 = accuracy(TOOLS_V1, "v1")
    print("v2 (refined):"); a2 = accuracy(TOOLS_V2, "v2")
    print("refinement changed accuracy from", a1, "to", a2, "out of", len(EVAL))
else:
    print("[skipped] expected: v2 routes more (often all) queries correctly than v1,")
    print("          because the refined descriptions draw clear, non-overlapping boundaries.")

**This cell:** the connection to the **Agent SDK**. These exact descriptions are what you
put in an `@tool` definition or an `AgentDefinition`; the routing signal is identical whether
the tool runs through the base API or the Agent SDK. Anthropic tool definitions also support an
optional `input_examples` field for well-formed example inputs, alongside the description.

In [ ]:
# ===== the same descriptions carry into the Agent SDK =====
print("base API tool:   {'name','description','input_schema'}  <- description drives routing")
print("Agent SDK @tool: @tool(name, description, schema)       <- same description, same effect")
print("Agent SDK agent: AgentDefinition(description=..., tools=[...])  scopes which tools it sees")

| anti-pattern | what to do instead |
|---|---|
| "get order info" on three tools | give each a distinct slice and say what it is for |
| rely on the tool name to disambiguate | put the routing signal in the description, not the name |
| omit when-NOT-to-use guidance | name the neighbours and say which tool to use instead |
| tune routing by rewriting code | rewrite the descriptions first; that is the real lever |

**Lesson:** the model routes on **descriptions**, so overlapping ones cause mis-routing no
matter how good the model is. Give each tool a clear boundary: its slice, its input, example
queries, and when NOT to use it. The overlap score and the accuracy test are two cheap ways to
catch ambiguity before your users do.

---

## Recap - descriptions drive selection

| Idea | In this lab | Course topic |
|---|---|---|
| Description as signal | vague v1 vs refined v2 | descriptions guide tool selection |
| Boundaries | input, examples, edge cases, when-not | clear tool boundaries |
| Ambiguity | overlap score high on v1 | overlapping descriptions cause mis-routing |
| Measurement | accuracy over ambiguous queries | test selection before vs after |

One principle to carry forward: **write each tool's description so the model can tell it apart
from its neighbours at a glance.** To run live, paste a real key into **Setup 2/3** and re-run
from the top. Then try it: add a fifth tool that overlaps `check_status`, and watch accuracy
drop until you give it a clear boundary. Next lab: scoping tools per agent and forcing tool use
with `tool_choice`.